In [19]:
##############################
###Model Configuration########
##############################
#cahnge detection head of yolo and use DetectionModel interface for using E2Eloss
#checking  libraries and GPU situtaions
import os
import glob
import sys

import torch
import torch.nn as nn
import torchvision.transforms as transforms

from ultralytics import YOLO

from model import Resnet50Yolo26

##setting configuration
os.environ['PYTORCH_NO_CUDA_MEMORY_CACHING'] ='1'
os.environ['CUDA_LAUNCH_BLOCKING']='1'
os.environ['TORCH_USE_CUDA_DSA']='1'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

#ignoring warnings
import warnings
warnings.filterwarnings('ignore')


In [20]:
#Defining Hyperparameters
EPOCHS=4
BATCHSIZE=1
LR=0.05 # learning rate

In [21]:
from pathlib import Path
from torch.utils.data import Dataset
from torchvision.io import decode_image

import os 
import glob
import shutil

import numpy as np

import time
import math

from typing import List

#class_index:x1,y1,x2,y2,x3,y3,x4,y4
class CustomDataset(Dataset):
    names = {
    '0': "plane",
    '1': "ship",
    '2': "storage tank",
    '3': "baseball diamond",
    '4': "tennis court",
    '5': "basketball court",
    '6': "ground track field",
    '7': "harbor",
    '8': "bridge",
    '9': "large vehicle",
    '10': "small vehicle",
    '11': "helicopter",
    '12': "roundabout",
    '13': "soccer ball field",
    '14': "swimming pool"
}
    def __init__(self,imagesDir='dataset/dota128/images',labelsDir='dataset/dota128/labels'
                 ,splitDir='train'):
        if  not os.path.exists(imagesDir) and os.path.exists(labelsDir):
            raise FileNotFoundError
        from ultralytics.data.converter import convert_dota_to_yolo_obb
        # convert_dota_to_yolo_obb()
        self.imagesParent=imagesDir #returns string
        self.labelsParent=labelsDir #returns string
        self.splitDir=splitDir #returns string

        self.imagesDir=os.listdir(os.path.join(self.imagesParent,self.splitDir))  #returns list
        
        # self.imagesDir=os.listdir(imagesDir) #returns list # we gotta problem three.# need to add new directory into current path(train,val)
        # self.labelsDir=os.listdir(labelsDir) #returns list #using glob.glob searching labels
        self.preprocessing=transforms.Compose(
            [transforms.Resize(size=(256,256)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])])
    def _xywhr_to_bbox(self,x, y, w, h, angle):
        """
        Convert rotated bounding box from:
            (cx, cy, w, h, angle)

        to four corner points:
            (x1, y1, x2, y2, x3, y3, x4, y4)

        angle is in radians.
        """

        # Half dimensions
        hw = w / 2
        hh = h / 2

        # Unrotated corners
        corners = torch.tensor([
            [-hw, -hh],
            [ hw, -hh],
            [ hw,  hh],
            [-hw,  hh]
        ], dtype=torch.float32)

        # Rotation matrix
        cos_a = math.cos(angle)
        sin_a = math.sin(angle)

        R = torch.tensor([
            [cos_a, -sin_a],
            [sin_a,  cos_a]
        ], dtype=torch.float32)

        # Rotate
        corners = corners @ R.T

        # Translate to center
        corners[:, 0] += x
        corners[:, 1] += y

        return corners
        
    def _dota_to_xywhr(self,labels) -> dict[str,torch.Tensor]:

        # we need to returns dict[str,torch.Tensor] for using V8OBBLoss
        converted = {}

        for label in labels:

            # ----------------------------------------------------
            # Class
            # ----------------------------------------------------

            cls = torch.tensor(label[0])
            converted['cls'] = cls
            

            # ----------------------------------------------------
            # Four points
            # --------------]]) #return--------------------------------------

            points = label[1:].reshape(4, 2)

            # x coordinates
            x = points[:, 0]

            # y coordinates
            y = points[:, 1]


            # ----------------------------------------------------
            # Center
            # ----------------------------------------------------

            cx = torch.tensor(x.mean())
            converted['cx'] = cx
            cy = torch.tensor(y.mean())
            converted['cy'] = cy


            # ----------------------------------------------------
            # Calculate edge vectors
            # ----------------------------------------------------

            edge1 = points[1] - points[0]
            edge2 = points[2] - points[1]


            # list----------------------------------------------------
            # Edge lengths
            # ------'----------------------------------------------

            len1 = torch.linalg.vector_norm(edge1)
            len2 = torch.linalg.vector_norm(edge2)


            # ----------------------------------------------------
            # Select longer edge as width
            # ----------------------------------------------------

            if len1 >= len2:

                width = len1
                height = len2

                angle = torch.atan2(
                    edge1[1],
                    edge1[0]
                )

            else:

                width = len2
                height = len1

                angle = torch.atan2(
                    edge2[1],
                    edge2[0]
                )
            converted['width'] = torch.as_tensor(
                width, dtype=torch.float32
            )

            converted['height'] = torch.as_tensor(
                height, dtype=torch.float32
            )

            converted['angle'] = torch.as_tensor(
                angle, dtype=torch.float32
            )

            converted['bboxes'] = torch.stack([
                converted['cx'],
                converted['cy'],
                converted['width'],
                converted['height'],
                converted['angle']
            ]).float()
            # ----------------------------------------------------
            # Save
            # ----------------------------------------------------

            # converted.append(
            #     torch.stack(
            #         [
            #             cls,
            #             cx,
            #             cy,
            #             width,
            #             height,
            #             angle
            #         ]
            #     )
            # )


        # --------------------------------------------------------
        # No objects
        # --------------------------------------------------------

        # if len(converted) == 0:

        #     return torch.empty(
        #         (0, 6),
        #         dtype=labels.dtype
        #     )


        return converted


    def _splittingDataset(self,path:Path|str):
        #I dont need this method
        if not os.path.exists(path):
            raise FileNotFoundError

        for val in os.listdir(path):
            
            parentPath=os.path.join(path,val)
            fullPath=os.path.join(parentPath,'train')

            dirs=os.listdir(fullPath)
            trains=dirs[:len(dirs)*0.7]  #returns List[str]
            vals=dirs[len(dirs)*0.7:] #returns List[str]


            #creating new files parent directory called val
            os.makedirs(os.path.join(parentPath,'val'),exist_ok=True)

            ##moving current images into val folders
            #traversing through val dir
            for val in vals:
                valPath=os.path.join(fullPath,val)
                #moving current file into val file
                if os.path.exists(valPath):
                    start=time.time()
                    shutil.move(
                        valPath,os.path.join(parentPath,'val')
                    )
                    end=time.time()
                    print("File sends within {} ms ".format(end-start))
            


    # def __getitem__(self, index):
    #     # return super().__getitem__(index)
    #     for val in ['train','val']:
    #         pass
    #     fullPath=os.path.join(self.imagesParent,self.imagesDir[index])
    #     # print(fullPath)
    #     self.labels=[]
    #     if os.path.exists(fullPath):
    #         #searching curent images on current path

    #         # print(self.imagesDir[index])
    #         #image preprocessing
    #         image = transforms.Todecode_imagePILImage()(decode_image(fullPath))
    #         image = self.preprocessing(image)           
    #         # print(self.imagesDir[index])

    #         self.labelsDir='.'.join([self.imagesDir[index].split('.')[:-1][0],'txt'])
    #         # print(self.imagesDir[index])
    #         labelFullPath=os.path.join(self.labelsParent,self.labelsDir)
    #         # print(labelFullPath)
    #         ##opening label file and extracting labels
    #         with open(labelFullPath,mode='r',encoding='utf-8') as file:
    #             #extracting labels
    #             # output=file.read()
    #             for ele in file.readlines():
    #                 # self.labels.append(ele)
    #                 list_=ele.split(' ')
    #                 list_[-1]=list_[-1][:-1]
    #                 # list_=lambda k[index]=ey:int(key)
    #                 list_=list(map(float,list_))
    #                 self.labels.append(list_)
    #           return (image,self.labels)
     
    def __getitem__(self,index):
            #preprocessing
            
            
            fullPath=os.path.join(self.imagesParent,self.splitDir,self.imagesDir[index])
            #preprocessing current images
            image=transforms.ToPILImage()(decode_image(fullPath))
            image=self.preprocessing(image)

            if os.path.exists(fullPath):
                self.labels=[]

                self.labelsDir='.'.join([self.imagesDir[index].split('.')[:-1][0],'txt']) #returns string

                labelFullPath=os.path.join(self.labelsParent,self.splitDir,self.labelsDir)

                with open(labelFullPath,mode='r',encoding='utf-8') as file:
                    #extracting labels
                    for ele in file.readlines():
                        ele=ele.strip()
                        list_=ele.split(' ')
                        list_=list(map(float,list_))
                        #converting current features to yolo prune format
                        list_=self._dota_to_xywhr(torch.tensor([list_])) #return torch.Tensor
                        # print(list_[1])
                        # print(list_)
                        self.labels.append(list_)
                        # print(list_)
                        

                return (image,self.labels)
    def __len__(self):
            return len(self.imagesDir)

In [22]:
#checking dataset paths
import os 
print(os.listdir('dataset/dota128/images/train'))

['P1333__1024__1048___2976.jpg', 'P2558__1024__3144___524.jpg', 'P2005__1024__1572___2620.jpg', 'P0254__682__349___698.jpg', 'P1652__2048__1952___1048.jpg', 'P2627__1024__0___0.jpg', 'P1845__1024__524___2096.jpg', 'P1393__1024__3668___5240.jpg', 'P1443__682__1396___1396.jpg', 'P1868__682__698___349.jpg', 'P2465__682__698___698.jpg', 'P0827__682__0___349.jpg', 'P2764__682__2094___2443.jpg', 'P2506__1024__907___524.jpg', 'P0907__682__0___698.jpg', 'P1299__1024__2620___1572.jpg', 'P1171__2048__3144___0.jpg', 'P1016__682__1396___421.jpg', 'P2226__682__2443___520.jpg', 'P1462__1024__1048___3668.jpg', 'P1357__1024__0___524.jpg', 'P0908__2048__383___454.jpg', 'P1360__682__2443___3318.jpg', 'P1140__2048__2096___2651.jpg', 'P2416__682__1167___349.jpg', 'P1059__682__2094___698.jpg', 'P1866__682__0___2443.jpg', 'P2471__2048__0___0.jpg', 'P0310__682__698___1047.jpg', 'P2250__1024__2041___524.jpg', 'P2727__682__975___769.jpg', 'P2676__682__1784___2964.jpg', 'P2739__1024__2096___524.jpg', 'P1867__10

In [23]:
#testing train dataset
from torch.utils.data import DataLoader
dataset=CustomDataset(splitDir='train')

print(dataset[0])

(tensor([[[-1.1418, -1.0048, -1.0733,  ..., -1.4843, -1.4843, -1.4843],
         [-0.8849, -0.9020, -0.9705,  ..., -1.4843, -1.4843, -1.4843],
         [-0.9363, -1.0562, -0.9534,  ..., -1.4843, -1.4843, -1.4843],
         ...,
         [-1.4672, -1.4672, -1.4500,  ..., -0.8335, -0.8335, -0.9877],
         [-1.4672, -1.4672, -1.4500,  ..., -0.8335, -0.8335, -1.0904],
         [-1.4672, -1.4672, -1.4500,  ..., -0.9192, -1.0048, -1.0733]],

        [[-1.0378, -0.8978, -0.9678,  ..., -1.3880, -1.3880, -1.3880],
         [-0.7752, -0.7927, -0.8627,  ..., -1.3880, -1.3880, -1.3880],
         [-0.8277, -0.9503, -0.8452,  ..., -1.3880, -1.3880, -1.3880],
         ...,
         [-1.3704, -1.3704, -1.3529,  ..., -0.7227, -0.7227, -0.8803],
         [-1.3704, -1.3704, -1.3529,  ..., -0.7227, -0.7227, -0.9853],
         [-1.3704, -1.3704, -1.3529,  ..., -0.8102, -0.8978, -0.9678]],

        [[-0.8110, -0.6715, -0.7413,  ..., -1.1596, -1.1596, -1.1596],
         [-0.5495, -0.5670, -0.6367,  ..., -

In [24]:
#testing val dataset
dataset=CustomDataset(splitDir='val')
print(dataset[0])


(tensor([[[-1.1418, -1.0048, -1.0733,  ..., -1.4843, -1.4843, -1.4843],
         [-0.8849, -0.9020, -0.9705,  ..., -1.4843, -1.4843, -1.4843],
         [-0.9363, -1.0562, -0.9534,  ..., -1.4843, -1.4843, -1.4843],
         ...,
         [-1.4672, -1.4672, -1.4500,  ..., -0.8335, -0.8335, -0.9877],
         [-1.4672, -1.4672, -1.4500,  ..., -0.8335, -0.8335, -1.0904],
         [-1.4672, -1.4672, -1.4500,  ..., -0.9192, -1.0048, -1.0733]],

        [[-1.0378, -0.8978, -0.9678,  ..., -1.3880, -1.3880, -1.3880],
         [-0.7752, -0.7927, -0.8627,  ..., -1.3880, -1.3880, -1.3880],
         [-0.8277, -0.9503, -0.8452,  ..., -1.3880, -1.3880, -1.3880],
         ...,
         [-1.3704, -1.3704, -1.3529,  ..., -0.7227, -0.7227, -0.8803],
         [-1.3704, -1.3704, -1.3529,  ..., -0.7227, -0.7227, -0.9853],
         [-1.3704, -1.3704, -1.3529,  ..., -0.8102, -0.8978, -0.9678]],

        [[-0.8110, -0.6715, -0.7413,  ..., -1.1596, -1.1596, -1.1596],
         [-0.5495, -0.5670, -0.6367,  ..., -

In [25]:
def collate_fn(batch):
    images, labels = zip(*batch)

    images = torch.stack(images, dim=0)

    return images, list(labels)

In [26]:
#preparing MNIST dataset 
#importing MNIST dataset from torch
from torchvision.datasets import MNIST
#importing DataLoader in torch
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# preprocessing=transforms.Compose([transforms.ToTensor(),transforms.Resize(size=256),transforms.Grayscale(num_output_channels=3)])
# preprocessing=transforms.Compose(
#     [transforms.Resize(size=(256,256)),
#      transforms.Grayscale(num_output_channels=3),
#      transforms.ToTensor(),
#      transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
#      ]
# )decode_image
#  transforms.Lambda(lambda x:x.convert('RGB'))
# dataset=MNIST(root='./MNISTDataset',download=True)
# print(dataset)
trainDataset=CustomDataset(splitDir='train')
trainDataLoader=DataLoader(trainDataset,batch_size=BATCHSIZE,shuffle=True,num_workers=0,collate_fn=collate_fn)

testDataset=CustomDataset(splitDir='val')
testDataLoader=DataLoader(testDataset,batch_size=BATCHSIZE,shuffle=True,num_workers=0,collate_fn=collate_fn)

#testing current dataset
display(next(iter(trainDataLoader)))
display(type(next(iter(trainDataLoader))))
##########################################
print("end===")
print()
print("testing val dataset")
print()
##testing##
display(next(iter(testDataLoader)))
display(type(next(iter(testDataLoader))))
####################################


(tensor([[[[-2.1008, -2.1008, -2.1008,  ..., -2.1008, -2.1008, -2.1008],
           [-2.1008, -2.1008, -2.1008,  ..., -2.1008, -2.1008, -2.1008],
           [-2.1008, -2.1008, -2.1008,  ..., -2.1008, -2.1008, -2.1008],
           ...,
           [ 1.1358,  1.1187,  1.1187,  ..., -0.0458, -0.0458, -0.1486],
           [ 1.1187,  1.1187,  1.1015,  ..., -0.0458,  0.1426,  0.0227],
           [ 1.1187,  1.1187,  1.1358,  ..., -0.0801,  0.0398, -0.0287]],
 
          [[-2.0182, -2.0182, -2.0182,  ..., -2.0182, -2.0182, -2.0182],
           [-2.0182, -2.0182, -2.0182,  ..., -2.0182, -2.0182, -2.0182],
           [-2.0182, -2.0182, -2.0182,  ..., -2.0182, -2.0182, -2.0182],
           ...,
           [ 1.2906,  1.2731,  1.2731,  ...,  0.0826,  0.0826, -0.0224],
           [ 1.2731,  1.2731,  1.2556,  ...,  0.0826,  0.2752,  0.1527],
           [ 1.2731,  1.2731,  1.2906,  ...,  0.0476,  0.1702,  0.1001]],
 
          [[-1.7870, -1.7870, -1.7870,  ..., -1.7870, -1.7870, -1.7870],
           [-

tuple

end===

testing val dataset



(tensor([[[[-0.4397, -0.4568, -0.4739,  ..., -0.8678, -0.8849, -0.7993],
           [-0.2171, -0.4397, -0.4739,  ..., -0.8678, -0.9192, -0.8164],
           [-0.0629, -0.1828, -0.3712,  ..., -0.9363, -0.8849, -0.7822],
           ...,
           [-0.6452, -0.6452, -0.6281,  ..., -0.9534, -1.0219, -1.1247],
           [-0.6281, -0.6281, -0.5596,  ..., -0.9534, -0.9192, -0.9705],
           [-0.6623, -0.6281, -0.6452,  ..., -0.9192, -0.8335, -0.7650]],
 
          [[-0.3200, -0.3375, -0.3550,  ..., -0.7577, -0.7752, -0.6877],
           [-0.0924, -0.3200, -0.3550,  ..., -0.7577, -0.8102, -0.7052],
           [ 0.0651, -0.0574, -0.2500,  ..., -0.8277, -0.7752, -0.6702],
           ...,
           [-0.5301, -0.5301, -0.5126,  ..., -0.8452, -0.9153, -1.0203],
           [-0.5126, -0.5126, -0.4426,  ..., -0.8452, -0.8102, -0.8627],
           [-0.5476, -0.5126, -0.5301,  ..., -0.8102, -0.7227, -0.6527]],
 
          [[-0.0964, -0.1138, -0.1312,  ..., -0.5321, -0.5495, -0.4624],
           [ 

tuple

In [27]:
image, labels = trainDataset[0]

print(type(image))
print(type(labels))
print(labels)

<class 'torch.Tensor'>
<class 'list'>
[{'cls': tensor(8.), 'cx': tensor(0.5637), 'cy': tensor(0.5798), 'width': tensor(0.6698), 'height': tensor(0.0363), 'angle': tensor(0.2475), 'bboxes': tensor([0.5637, 0.5798, 0.6698, 0.0363, 0.2475])}, {'cls': tensor(8.), 'cx': tensor(0.1191), 'cy': tensor(0.3794), 'width': tensor(0.0493), 'height': tensor(0.0173), 'angle': tensor(1.7099), 'bboxes': tensor([0.1191, 0.3794, 0.0493, 0.0173, 1.7099])}]


In [28]:
##POST TRAINING CURRENT MODEL
#Model training witgh MNIST dataset
import importlib
import model as model_module

from torch.nn.parallel import DataParallel,DistributedDataParallel


importlib.reload(model_module)

Resnet50Yolo26=model_module.Resnet50Yolo26

from collections import OrderedDict
import tqdm
model=Resnet50Yolo26(yoloWeights='yolo26n.pt').to(device)


In [29]:
import torch.nn as nn
from ultralytics.utils.loss import v8OBBLoss,BboxLoss


# 1. Define a dummy wrapper that looks exactly like a YOLO model to the loss function
class YOLOLossMock(nn.Module):
    def __init__(self, obb_head):
        super().__init__()
        # The loss looks for model.model[-1] to find the head
        self.model = nn.ModuleList([obb_head])
        
        # Copy critical metadata up to the root level
        # self.stride = obb_head.stride if hasattr(obb_head, 'stride') else getattr(obb_head, 'strides', None)
        self.stride=obb_head.stride
        self.nc = obb_head.nc
        self.no = obb_head.no
        
        # If your head has an args dictionary, map it; otherwise, provide an empty dict
        self.args = getattr(obb_head, 'args', {})

# 2. Extract your raw head from the composite model
# (Ensuring any DDP/DP wrapper on the composite is stripped out first)
base_composite = model.module if hasattr(model, 'module') else model
raw_obb_head = base_composite.detectionHead  # Adjust this if your head is nested differently

# 3. Wrap the head in our mock class
mock_yolo_model = YOLOLossMock(raw_obb_head)


# criterion = v8OBBLoss(model=mock_yolo_model, tal_topk=10)

In [30]:
for params in model.parameters():
    params.requires_grad=True

In [31]:
##defining loss and optimizer function
#importing required libraries
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

from ultralytics.utils.loss import v8OBBLoss,BboxLoss
from ultralytics.utils.loss import E2ELoss


from types import SimpleNamespace


optimizer=Adam(
    model.parameters(),lr=LR
)
# criterion=CrossEntropyLoss()
# criterion=BboxLoss()
criterion=v8OBBLoss(model=mock_yolo_model, tal_topk=10)
criterion.hyp = SimpleNamespace(
    box=7.5,
    cls=0.5,
    dfl=1.5,
    angle=1.06
)

for epoch in range(EPOCHS):
    for index,(data,targets) in enumerate(tqdm.tqdm(trainDataLoader)):
        #passing data into device
        data=data.to(device)
        output=model(data)

        #configure the detection head for the loss function
        # head=model.detectionHead
        # head.stride=torch.tensor(
        #     [4.,8.,16.,32.],
        #     device=data.device
        # ) #we gotta problem there
        # if head.stride
       
            # ---------------------------------
        # Convert dataset targets
        # ---------------------------------
        labels = targets[0]   # list of object dictionaries

        loss_target = {
            "cls": torch.stack([
                x["cls"] for x in labels
            ]).float().to(device),

            "bboxes": torch.stack([
                x["bboxes"] for x in labels
            ]).float().to(device),

            "batch_idx": torch.zeros(
                len(labels),
                dtype=torch.long,
                device=device
            )
        }
        ##Backward pass

        lossItems,lossDetails=criterion.loss(output,loss_target)

        totalLoss=lossItems.sum()

        ##Backward Pass###
        optimizer.zero_grad()
        totalLoss.backward()

        optimizer.step()


100%|██████████| 128/128 [00:14<00:00,  8.62it/s]


In [ ]:
# In[ ]:


###Calculating detection with IoU(Intersection over Union)####
##importing reqyuired libraries
import torch
import tqdm
from ultralytics.utils.metrics import box_iou


#setting model into evaluatuion mode
model.eval()
with torch.no_grad():
    for index,(data,targets) in enumerate(tqdm.tqdm(testDataLoader)):
        ##forward pass
        data=data.to(device)
        output=model(data)

        #configure curent targets for data
        labels=targets[0]  # list of object dictionaries
        target = {
            "cls": torch.stack([
                x["cls"] for x in labels
            ]).float().to(device),

            "bboxes": torch.stack([
                x["bboxes"] for x in labels
            ]).float().to(device),

            "batch_idx": torch.zeros(
                len(labels),
                dtype=torch.long,
                device=device
            )
        }


        scores = output[1]["scores"]

        probs = scores.sigmoid()
        confidence = probs.max(dim=1).values

        print("Mean confidence:", confidence.mean().item())
        print("Max confidence:", confidence.max().item())

  3%|▎         | 4/128 [00:00<00:07, 17.17it/s]

Mean confidence: 2.993638403836485e-08
Max confidence: 8.293571590911597e-05
Mean confidence: 4.950562529359992e-11
Max confidence: 1.8579711102972851e-09
Mean confidence: 1.1842561375985383e-09
Max confidence: 1.1343147434672574e-06
Mean confidence: 4.9359048098773783e-11
Max confidence: 1.8570747162272028e-09
Mean confidence: 5.016653412126537e-11
Max confidence: 1.8797718936980345e-09


  7%|▋         | 9/128 [00:00<00:06, 18.51it/s]

Mean confidence: 0.001661596237681806
Max confidence: 0.9460724592208862
Mean confidence: 5.7304216927178686e-11
Max confidence: 2.923691688749841e-08
Mean confidence: 4.95224590502108e-11
Max confidence: 1.855449460741454e-09
Mean confidence: 0.0013342984020709991
Max confidence: 0.9172409772872925


 10%|█         | 13/128 [00:00<00:06, 18.40it/s]

...

Mean confidence: 2.3934489945531823e-08
Max confidence: 0.00011773998994613066
Mean confidence: 2.1429077605716884e-05
Max confidence: 0.016164835542440414
Mean confidence: 5.135646768961166e-11
Max confidence: 4.120603414037305e-09


 99%|█████████▉| 127/128 [00:08<00:00, 15.35it/s]

Mean confidence: 4.9329585555257793e-11
Max confidence: 1.849672415232817e-09
Mean confidence: 5.0960763386953545e-11
Max confidence: 1.850660402702431e-09
Mean confidence: 6.56478413296746e-11
Max confidence: 1.1538341482264514e-07
Mean confidence: 4.943773515564409e-11
Max confidence: 1.8578436566940582e-09


100%|██████████| 128/128 [00:08<00:00, 15.88it/s]

Mean confidence: 1.5100909411813745e-10
Max confidence: 2.1022688656557875e-07
